# Handwritten Digit Classification with a CNN

Build and evaluate a convolutional neural network (CNN) that recognizes handwritten digits from the MNIST dataset. This notebook is designed to run from top to bottom in Google Colab; the dataset is downloaded automatically.

**Learning goals**
- Prepare image data for a neural network.
- Build and train a CNN with TensorFlow/Keras.
- Evaluate the model and inspect its predictions.

In Colab, optionally choose **Runtime → Change runtime type → T4 GPU** to speed up training.

## 1. Import libraries and set a random seed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
print("TensorFlow version:", tf.__version__)

## 2. Load and explore MNIST

MNIST contains 28 × 28 grayscale images of handwritten digits (0–9). Keras downloads the training and test sets the first time this cell runs.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Training images: {x_train.shape}; training labels: {y_train.shape}")
print(f"Test images: {x_test.shape}; test labels: {y_test.shape}")

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, x_train[:10], y_train[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Prepare the images

Pixel intensities range from 0 to 255. Scaling them to 0–1 helps neural networks train more reliably. The CNN expects a channel dimension, so reshape each image to `(28, 28, 1)`.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("Prepared training shape:", x_train.shape)
print("Prepared test shape:", x_test.shape)

## 4. Build the CNN

Convolution and pooling layers learn visual patterns such as edges and loops. The final dense layer returns one score for each digit class.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Conv2D(64, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. Train the model

A portion of the training data is held out for validation so we can monitor performance on examples the model is not learning from directly.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on the test set

The test set was not used during training or validation. Its accuracy gives a better estimate of how the model performs on new digit images.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")

In [ ]:
probabilities = model.predict(x_test, verbose=0)
predictions = np.argmax(probabilities, axis=1)
confusion = tf.math.confusion_matrix(y_test, predictions, num_classes=10).numpy()

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(confusion, cmap="Blues")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
threshold = confusion.max() / 2
for row in range(10):
    for col in range(10):
        ax.text(col, row, confusion[row, col], ha="center", va="center",
                color="white" if confusion[row, col] > threshold else "black")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

## 7. Inspect individual predictions

Correct and incorrect examples are both useful: incorrect predictions can reveal which digits the model confuses.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for index, ax in enumerate(axes.flat):
    ax.imshow(x_test[index].squeeze(), cmap="gray")
    predicted = predictions[index]
    actual = y_test[index]
    ax.set_title(f"Predicted: {predicted} | Actual: {actual}",
                 color="green" if predicted == actual else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Try it yourself

1. Change the number of epochs or batch size. How does training time and test accuracy change?
2. Add or remove a convolutional layer. Does the model improve, or does it begin to overfit?
3. Pick a digit the model misclassified and inspect its confidence scores with `probabilities[index]`.
4. Save the trained model with `model.save("mnist_cnn.keras")`.